In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================
# Task 2 — Tomato (10 classes)
# Robust to weight download errors (retries + local fallback)
# ============================

import os, shutil, random, time, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             precision_recall_fscore_support, roc_curve, roc_auc_score)
from sklearn.preprocessing import label_binarize
from scipy.stats import ttest_rel

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

# ---------- Auto-detect PlantVillage root ----------
BASE_DIR_CANDIDATES = [
    "/kaggle/input/plantdisease/PlantVillage",
    "/kaggle/input/plantdisease/color",
    "/kaggle/input/plantdisease",
    "/kaggle/input/plantvillage",
    "/kaggle/input/PlantVillage"
]
base_dir = None
for p in BASE_DIR_CANDIDATES:
    if os.path.exists(p):
        base_dir = p
        break
assert base_dir is not None, "Dataset not found. In Kaggle, Add Data → `emmarex/plantdisease`."

# Working directory where we write a clean split
work_root = "/kaggle/working/Tomato_Split"
os.makedirs(work_root, exist_ok=True)

# Reproducibility
random_seed = 1
random.seed(random_seed)
np.random.seed(random_seed)
tf.random.set_seed(random_seed)

# ---------- Splits (meet requirement: Test in [0.20, 0.30]) ----------
TEST_FRACTION = 0.25
VAL_FRACTION_OF_TRAINVAL = 0.20
assert 0.20 <= TEST_FRACTION <= 0.30, "TEST_FRACTION must be between 0.20 and 0.30."

# Image/batch
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training schedule (two-stage)
STAGE1_EPOCHS = 35   # feature extraction
STAGE2_EPOCHS = 40   # fine-tuning
UNFREEZE_FRACTION = 0.25  # unfreeze top 25% of base in stage 2

# Optim / LR policy
BASE_LR = 3e-4
MIN_LR = 1e-6
PATIENCE_ES = 3
PATIENCE_RLRP = 1

# Tomato classes (10 expected)
tomato_classes = [
     'Tomato_Bacterial_spot',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Leaf_Mold',
    'Tomato_Septoria_leaf_spot',
    'Tomato_Spider_mites_Two_spotted_spider_mite',
    'Tomato__Target_Spot',
    'Tomato__Tomato_YellowLeaf__Curl_Virus',
    'Tomato__Tomato_mosaic_virus',
    'Tomato_healthy'



]

print("base_dir:", base_dir, "| work_root:", work_root)

# =========================
# Robust pretrained loader
# =========================
WEIGHTS_DIR_CANDIDATES = [
    "/kaggle/input/keras-application-weights",
    "/kaggle/input/keras-weights",
    "/kaggle/input/tf-imagenet-weights",
]

WEIGHT_FILENAMES = {
    "VGG16": "vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",
    "ResNet50": "resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5",
    "EfficientNetB0": "efficientnetb0_notop.h5",  # common name used in many mirrors
}

def _find_local_weight_path(model_name):
    fname = WEIGHT_FILENAMES.get(model_name)
    if not fname:
        return None
    for d in WEIGHTS_DIR_CANDIDATES:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

def _try_imagenet_with_retries(base_fn, input_shape, n_retries=3, sleep_s=3):
    last_err = None
    for attempt in range(1, n_retries+1):
        try:
            base = base_fn(weights="imagenet", include_top=False, input_shape=input_shape)
            return base
        except Exception as e:
            last_err = e
            print(f"[retry {attempt}/{n_retries}] Failed to load ImageNet weights: {e}")
            time.sleep(sleep_s)
    raise last_err

def load_base_with_fallback(model_name, base_fn, input_shape):
    """
    Order:
      1) Try ImageNet with retries (internet path)
      2) Try local file (if attached)
      3) Fallback to random init (warning)
    Returns: (base_model, source_str)
    """
    # 1) Internet download (with retries)
    try:
        base = _try_imagenet_with_retries(base_fn, input_shape)
        print(f"[{model_name}] Using ImageNet weights (download/cache).")
        return base, "imagenet"
    except Exception as e:
        print(f"[{model_name}] WARN: ImageNet weights unavailable after retries: {e}")

    # 2) Local weights (optional dataset)
    local_path = _find_local_weight_path(model_name)
    if local_path is not None:
        try:
            base = base_fn(weights=local_path, include_top=False, input_shape=input_shape)
            print(f"[{model_name}] Using LOCAL weights: {local_path}")
            return base, "local"
        except Exception as e:
            print(f"[{model_name}] WARN: Local weights failed to load: {e}")

    # 3) Random init fallback
    base = base_fn(weights=None, include_top=False, input_shape=input_shape)
    print(f"[{model_name}] Using RANDOM initialization (no pretrained weights). "
          f"Tip: attach a weights dataset or keep Internet ON.")
    return base, "random"

# =========================
# Split builder (copy files)
# =========================
def prepare_splits(src_root, dest_root, classes, test_frac, val_frac_of_remaining, seed=123):
    random.seed(seed)
    for split in ['train','val','test']:
        for cls in classes:
            os.makedirs(os.path.join(dest_root, split, cls), exist_ok=True)

    for cls in classes:
        # locate class folder (some datasets nested)
        candidate = os.path.join(src_root, cls)
        found_dir = None
        if os.path.exists(candidate):
            found_dir = candidate
        else:
            for root, dirs, _ in os.walk(src_root):
                if os.path.basename(root) == cls:
                    found_dir = root
                    break

        if found_dir is None:
            print("⚠️ Missing folder, skipping:", cls)
            continue

        files = [f for f in os.listdir(found_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tif','.tiff'))]
        random.shuffle(files)
        n = len(files)
        if n == 0:
            print("⚠️ No images for:", cls)
            continue

        n_test = int(n * test_frac)
        n_val = int((n - n_test) * val_frac_of_remaining)

        test_files = files[:n_test]
        val_files = files[n_test:n_test+n_val]
        train_files = files[n_test+n_val:]

        for f in train_files:
            shutil.copy(os.path.join(found_dir,f), os.path.join(dest_root,'train',cls,f))
        for f in val_files:
            shutil.copy(os.path.join(found_dir,f), os.path.join(dest_root,'val',cls,f))
        for f in test_files:
            shutil.copy(os.path.join(found_dir,f), os.path.join(dest_root,'test',cls,f))

        print(f"{cls}: total={n} -> train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

def clean_empty_and_hidden_dirs(root_dir):
    for root_dirpath, dirs, files in os.walk(root_dir):
        for d in dirs:
            dir_path = os.path.join(root_dirpath, d)
            if d.startswith('.') or 'checkpoint' in d.lower():
                shutil.rmtree(dir_path, ignore_errors=True)
            else:
                try:
                    if os.path.isdir(dir_path) and len(os.listdir(dir_path)) == 0:
                        os.rmdir(dir_path)
                except Exception:
                    pass

prepare_splits(base_dir, work_root, tomato_classes, TEST_FRACTION, VAL_FRACTION_OF_TRAINVAL, random_seed)
clean_empty_and_hidden_dirs(work_root)

for s in ['train','val','test']:
    total = 0
    for c in tomato_classes:
        p = os.path.join(work_root, s, c)
        if os.path.exists(p):
            total += len(os.listdir(p))
    print(f"{s}: {total} images")

# =========================
# Generators
# =========================
clean_empty_and_hidden_dirs(work_root)

train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)
test_val_gen = ImageDataGenerator(rescale=1./255)

train_gen = train_aug.flow_from_directory(
    os.path.join(work_root,'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=random_seed
)
val_gen = test_val_gen.flow_from_directory(
    os.path.join(work_root,'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=random_seed
)
test_gen = test_val_gen.flow_from_directory(
    os.path.join(work_root,'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=random_seed
)

num_classes = train_gen.num_classes
class_names = list(train_gen.class_indices.keys())
print("Classes detected:", class_names)
print("Counts -> Train:", train_gen.samples, "Val:", val_gen.samples, "Test:", test_gen.samples)

# =========================
# Model head & callbacks
# =========================
def build_classifier(base_model, lr=BASE_LR, head_units=256):
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dropout(0.4),
        Dense(head_units, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def get_callbacks(prefix):
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=PATIENCE_RLRP,
                          min_lr=MIN_LR, verbose=1),
        ModelCheckpoint(f"/kaggle/working/{prefix}.h5", monitor='val_accuracy', save_best_only=True, verbose=1)
    ]

# =========================
# Two-stage training
# =========================
def two_stage_train(model_name, base_fn,
                    stage1_epochs=STAGE1_EPOCHS, stage2_epochs=STAGE2_EPOCHS,
                    unfreeze_fraction=UNFREEZE_FRACTION,
                    lr_stage1=BASE_LR, lr_stage2=1e-5):
    print(f"\n=== {model_name}: Stage 1 (feature extraction) ===")
    base, src = load_base_with_fallback(model_name, base_fn, IMG_SIZE+(3,))
    base.trainable = False
    model = build_classifier(base, lr=lr_stage1)
    cb1 = get_callbacks(f"{model_name}_stage1")
    hist1 = model.fit(train_gen, validation_data=val_gen, epochs=stage1_epochs, callbacks=cb1, verbose=1)

    print(f"=== {model_name}: Stage 2 (fine-tune top {int(unfreeze_fraction*100)}%) ===")
    total_layers = len(base.layers)
    unfreeze_at = int(total_layers * (1 - unfreeze_fraction))
    for i, layer in enumerate(base.layers):
        layer.trainable = True if i >= unfreeze_at else False

    model.compile(optimizer=Adam(learning_rate=lr_stage2), loss='categorical_crossentropy', metrics=['accuracy'])
    cb2 = get_callbacks(f"{model_name}_stage2")
    hist2 = model.fit(train_gen, validation_data=val_gen, epochs=stage2_epochs, callbacks=cb2, verbose=1)

    return model, hist1, hist2, src

# =========================
# Evaluation on TEST
# =========================
def evaluate_on_test(model, model_name, class_names):
    test_gen.reset()
    y_probs = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(y_probs, axis=1)
    y_true = test_gen.classes

    test_acc = accuracy_score(y_true, y_pred)

    labels_present = sorted(list(set(y_true) | set(y_pred)))
    target_names = [class_names[i] for i in labels_present]
    print("\n=== Classification Report (TEST) ===")
    print(classification_report(y_true, y_pred, labels=labels_present, target_names=target_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    plt.figure(figsize=(9,7))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_name} — Confusion Matrix (TEST)")
    plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout(); plt.show()

    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    try:
        auc_macro = roc_auc_score(y_true_bin, y_probs, average='macro', multi_class='ovr')
        auc_micro = roc_auc_score(y_true_bin, y_probs, average='micro', multi_class='ovr')
    except Exception:
        auc_macro, auc_micro = np.nan, np.nan

    # ROC plots (subset for readability)
    K = min(10, len(class_names))
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_probs.ravel())
    plt.figure(figsize=(8,6))
    plt.plot(fpr_micro, tpr_micro, linestyle=':', label=f"micro (AUC={auc_micro:.3f})")
    for i in range(K):
        try:
            fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            auc_i = roc_auc_score(y_true_bin[:, i], y_probs[:, i])
            plt.plot(fpr_i, tpr_i, lw=1, label=f"{class_names[i]} (AUC={auc_i:.3f})")
        except Exception:
            pass
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"{model_name} — ROC (TEST, first {K} classes) | macro AUC={auc_macro:.3f}")
    plt.legend(fontsize='x-small', bbox_to_anchor=(1.05,1))
    plt.tight_layout(); plt.show()

    # per-class accuracy
    class_acc = []
    for i in range(len(class_names)):
        idx = np.where(y_true == i)[0]
        if len(idx) == 0:
            class_acc.append(np.nan)
        else:
            class_acc.append(accuracy_score(y_true[idx], y_pred[idx]))

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "y_probs": y_probs,
        "overall_acc": test_acc,
        "auc_macro": auc_macro,
        "auc_micro": auc_micro,
        "confusion_matrix": cm,
        "class_accuracy": class_acc
    }

# =========================
# Train all three + TEST eval
# =========================
models_to_run = [
    ("VGG16", VGG16),
    ("ResNet50", ResNet50),
    ("EfficientNetB0", EfficientNetB0),
]

trained_models = {}
reports = {}
train_times_sec = {}
test_times_sec = {}
weight_sources = {}

for name, fn in models_to_run:
    print("\n" + "="*70)
    print(f"Training: {name}")
    t0 = time.time()
    model, h1, h2, src = two_stage_train(name, fn)
    t1 = time.time()
    train_times_sec[name] = t1 - t0
    weight_sources[name] = src
    trained_models[name] = model

    print("Evaluating on TEST set...")
    t0 = time.time()
    rep = evaluate_on_test(model, name, class_names)
    t1 = time.time()
    test_times_sec[name] = t1 - t0
    reports[name] = rep

    print(f"{name} — TEST Acc: {rep['overall_acc']:.4f} | AUC(macro): {rep['auc_macro']:.4f} | AUC(micro): {rep['auc_micro']:.4f} | weights={src}")

# =========================
# Summary tables + CSVs
# =========================
rows = []
for name, r in reports.items():
    rows.append({
        "Model": name,
        "Weights Source": weight_sources.get(name, "unknown"),
        "Test Accuracy": r["overall_acc"],
        "Test AUC (macro OVR)": r["auc_macro"],
        "Test AUC (micro)": r["auc_micro"],
        "Train Time (s)": train_times_sec.get(name, np.nan),
        "Test Eval Time (s)": test_times_sec.get(name, np.nan),
    })
summary_df = pd.DataFrame(rows).sort_values("Test Accuracy", ascending=False)
display(summary_df)

per_class_rows = []
for name, r in reports.items():
    y_true = r["y_true"]; y_pred = r["y_pred"]
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(class_names))), zero_division=0
    )
    for i, cls in enumerate(class_names):
        per_class_rows.append({
            "Model": name,
            "Class": cls,
            "Precision": precision[i],
            "Recall": recall[i],
            "F1": f1[i],
            "Support": support[i],
            "Class Accuracy": r["class_accuracy"][i]
        })
per_class_df = pd.DataFrame(per_class_rows)

summary_path  = "/kaggle/working/task2_tomato_TEST_summary_metrics.csv"
perclass_path = "/kaggle/working/task2_tomato_TEST_per_class_metrics.csv"
summary_df.to_csv(summary_path, index=False)
per_class_df.to_csv(perclass_path, index=False)
print("Saved:", summary_path)
print("Saved:", perclass_path)

# =========================
# Paired t-tests on TEST
# =========================
names = list(reports.keys())
correct = {m: (reports[m]["y_pred"] == reports[m]["y_true"]).astype(int) for m in names}

ttest_rows = []
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a = correct[names[i]]
        b = correct[names[j]]
        tstat, pval = ttest_rel(a, b)
        ttest_rows.append({
            "Model A": names[i],
            "Model B": names[j],
            "t-stat": float(tstat),
            "p-value": float(pval)
        })
ttest_df = pd.DataFrame(ttest_rows)
display(ttest_df)

ttest_path = "/kaggle/working/task2_tomato_TEST_paired_ttest.csv"
ttest_df.to_csv(ttest_path, index=False)
print("Saved:", ttest_path)
